<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **11. Local LLM meets PID**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 11.1 Introduction

[04.2 PID Controller](../04_Mobile_Algorithms/04.2_PID_Controller.ipynb) introduced how proportional, integral, and derivative control shape a system's response to a target change. This notebook builds on that foundation to explore another role for AI: supporting the interpretation of control experiments.

The experiment combines a simulated PID-controlled system with a pretrained large language model (LLM) running locally on the course Raspberry Pi. Adjusting the PID gains produces different responses, which Python summarises through measurements such as overshoot, final error, and settling time. The LLM receives these measurements and explains the response in words. The simulation and measurements run on the notebook computer; the Pi provides the AI inference. The PID controller continues to determine the control input, while the LLM acts as an assistant for interpreting the evidence.

The purpose is to connect a numerical experiment to a local AI model and assess whether its explanation is supported by the data. Comparing two gain combinations provides a basis for checking the model's claims against plots and measurements, and for distinguishing observed behavior from explanations that require further experiments. Requesting an explanation uses the model's existing training; it does not train a new controller or establish that the system is stable.

The prerequisites are the PID concepts in Notebook 04.2 and the Raspberry Pi connection and model setup described in [05.04 Getting Started with Raspberry Pi](05.04_GettingStarted_with_Raspberry_Pi.ipynb). The following sections cover the simulation and explanation workflow. The final questions guide a critical assessment of the AI's contribution to the analysis.


### 11.2 Simulate a PID Response

Start with [04.2 PID Controller](../04_Mobile_Algorithms/04.2_PID_Controller.ipynb) for the control theory and tuning exercises, especially Section 2.2.3 for the full PID sliders and Section 2.4 for choosing the gains. Here you will use a pretrained LLM running on the course Raspberry Pi to interpret a simulated response.

Run the cell below and adjust the P, I, and D sliders. The simulated plant is a mass–spring–damper system, initially at rest, with a target that steps from 0 to 10 at 10 seconds. The latest slider run is saved for the explanation in Section 11.3.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO

def simulate_pid_response(Kp=1.0, Ki=0.0, Kd=0.0):
    t = np.arange(0, 150.0, 0.005)
    dt = t[1] - t[0]
    
    setpoint_final = 10.0  # desired final value
    setpoint_time = 10.0    # time when setpoint jumps
    x = 0.0                 # position
    v = 0.0                 # velocity
    integral = 0.0
    previous_error = 0.0
    values, errors, integrals, setpoints = [], [], [], []

    for ti in t:
        # Step input: setpoint stays 0 until setpoint_time
        setpoint = setpoint_final if ti >= setpoint_time else 0.0
        error = setpoint - x
        integral += error * dt
        derivative = (error - previous_error) / dt
        u = Kp * error + Ki * integral + Kd * derivative

        # Mass-spring-damper: d²x/dt² = -x - 0.2*dx/dt + u
        dx = v
        dv = -x - 0.2*v + u
        x += dx * dt
        v += dv * dt

        values.append(x)
        errors.append(error)
        integrals.append(integral)
        setpoints.append(setpoint)
        previous_error = error

    # Keep this exact slider run for the explanation in Section 11.3.
    global pid_last_run
    pid_last_run = {
        "time": t.copy(), "position": np.asarray(values),
        "target": setpoint_final, "step_time": setpoint_time,
        "gains": {"Kp": Kp, "Ki": Ki, "Kd": Kd},
    }

    # Plot the response
    with plt.ioff():
        fig = plt.figure(figsize=(10,4))
    plt.plot(t, values, label="System Output (x)")
    plt.plot(t, setpoints, 'r--', label="Setpoint")
    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.ylim(top=15.0)  # Keep the y-axis maximum fixed for every slider setting.
    plt.legend()
    plt.grid(True)
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format="png", dpi=100)
            pid_image.value = buffer.getvalue()
    finally:
        plt.close(fig)

# Retire this graph's old observers and widgets when the cell is rerun.
if "pid_controls" in globals():
    for control in pid_controls.values():
        control.unobserve(pid_changed, names="value")
        control.close()
    pid_ui.close()
    pid_image.close()

pid_controls = {
    "Kp": widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description="Kp", continuous_update=False),
    "Ki": widgets.FloatSlider(value=0.08, min=0.0, max=0.2, step=0.001, description="Ki", continuous_update=False),
    "Kd": widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description="Kd", continuous_update=False),
}
pid_image = widgets.Image(format="png")
pid_image.layout.max_width = "100%"

def pid_changed(change=None):
    simulate_pid_response(**{name: control.value for name, control in pid_controls.items()})

for control in pid_controls.values():
    control.observe(pid_changed, names="value")
pid_changed()
pid_ui = widgets.VBox([*pid_controls.values(), pid_image])
display(pid_ui)


### 11.3 Explain the PID Response with an LLM

A **large language model (LLM)** can help interpret the response you obtained in Section 11.2. Python calculates the measurements, and the LLM explains them in words. The PID equations still determine every control input.

We use an existing pretrained model, so **no additional training is needed**. Each request supplies the current gains, a description of the simulated system, and measurements from the latest full PID plot. Like the advisor in [05.10 Local RAG meets Tit-for-Tat](05.10_Local_RAG_meets_Tit-for-Tat.ipynb), the model interprets evidence. Here we provide that evidence directly, so, no document retrieval or knowledge cards are needed.

Python measures overshoot, the final error (target minus final position), and settling time within 2% of the target, measured from the target change. Settling only means staying within that band for the rest of the recorded run. We also report the position range during the final 10 seconds and the number of crossings between errors outside opposite sides of the tolerance band.

Connect the course Raspberry Pi as described in [05.04 Getting Started with Raspberry Pi](05.04_GettingStarted_with_Raspberry_Pi.ipynb). The explanation uses the same `caspi` connection and `granite4.2:3b` model as the Tit-for-Tat advisor. The simulations and measurements run in this notebook but the LLM runs on the Pi.

Run the following cell, then click **Explain this response**. Change the full PID sliders in Section 11.2 and click again to explain the new run. Compare each explanation with the plot and measurements: a fluent explanation can still be wrong, and the fixed plot limits can hide large excursions.


In [ ]:
import json
import sys
from pathlib import Path

# Find the existing course connection from either the repo or notebook folder.
_caspi_dir = next(
    root / "02_Notebooks/05_Mobile_AI"
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "02_Notebooks/05_Mobile_AI/caspi.py").is_file()
)
if str(_caspi_dir) not in sys.path:
    sys.path.insert(0, str(_caspi_dir))


def summarise_pid_response(run):
    time = run["time"]
    position = run["position"]
    if not np.all(np.isfinite(position)):
        raise ValueError("This run contains non-finite values. Reduce the gains and rerun it.")
    after_step = time >= run["step_time"]
    times, positions = time[after_step], position[after_step]
    target = run["target"]
    error = target - positions
    tolerance = 0.02 * abs(target)
    outside = np.flatnonzero(np.abs(error) > tolerance)
    settled_at = (0.0 if outside.size == 0 else
                  None if outside[-1] == len(times) - 1 else
                  float(times[outside[-1] + 1] - run["step_time"]))
    late = positions[times >= times[-1] - 10.0]
    # Ignore crossings that stay inside the tolerance band.
    signs = np.sign(error[np.abs(error) > tolerance])
    return {
        "gains": run["gains"],
        "target_position": float(target),
        "target_step_time_s": float(run["step_time"]),
        "last_recorded_time_s": float(times[-1]),
        "sampling_interval_s": float(time[1] - time[0]),
        "overshoot_percent": float(max(0.0, (positions.max() - target) / abs(target) * 100)),
        "final_error_target_minus_position": float(error[-1]),
        "settling_band_position_units": float(tolerance),
        "settling_time_after_step_s": settled_at,
        "position_min_after_step": float(positions.min()),
        "position_max_after_step": float(positions.max()),
        "position_range_last_10_s": float(np.ptp(late)),
        "target_crossings_outside_band": int(np.count_nonzero(np.diff(signs))),
    }


# Replace the previous explanation widgets when this cell is rerun.
if "pid_explain_button" in globals():
    pid_explain_button.on_click(explain_pid_response, remove=True)
    pid_explain_button.close()
    pid_explain_output.close()
    pid_explain_ui.close()

pid_explain_button = widgets.Button(description="Explain this response", layout=widgets.Layout(width="auto"))
pid_explain_output = widgets.Output()


def explain_pid_response(_):
    pid_explain_button.disabled = True
    with pid_explain_output:
        pid_explain_output.clear_output(wait=True)
        try:
            if "pid_last_run" not in globals():
                print("Run the full PID slider cell in Section 11.2 first.")
                return
            measurements = summarise_pid_response(pid_last_run)
            evidence = json.dumps(measurements, indent=2, allow_nan=False)
            print("Measurements for this explanation (null means not settled during the run):")
            print(evidence)
            prompt = (
                "Explain this educational PID simulation in at most 200 words. "
                "The plant is x'' = -x - 0.2*x' + u, initially at rest at x=0. "
                "PID uses error = target - position, a summed integral, and a backward-difference derivative. "
                "The simulation uses forward Euler. There are no actuator limits, delays, or sensor noise. "
                "Position uses generic simulation units; this is not an insulin or other medical model. "
                "The target steps from zero to the reported positive target. "
                "Use the supplied measurements as evidence; you have not been given the plot. "
                "Describe overshoot, remaining error, settling, and variation. "
                "Cite the relevant numbers and relate them cautiously to P, I, and D. "
                "A null settling time means not settled within the recorded window. "
                "Distinguish measured observations from possible explanations. "
                "Do not equate a small final error with settling or claim proven stability. "
                "Do not invent faults, measurements, or an optimal gain combination.\n\n"
                "MEASUREMENTS:\n" + evidence
            )
            import caspi
            print("\nAsking the local LLM; loading the model may take a few minutes...", flush=True)
            reply = caspi.ask(
                "granite4.2:3b", prompt, think=False, temperature=0.2,
                num_predict=500, stream=False, show=False, stats=False,
            )
            print("\nLLM interpretation:\n" + reply.text)
        except (ImportError, RuntimeError, ValueError) as error:
            print(f"Explanation unavailable: {error}")
            print("Check the Pi connection and model setup in Notebook 05.04, then click again.")
        finally:
            pid_explain_button.disabled = False


pid_explain_button.on_click(explain_pid_response)
pid_explain_ui = widgets.VBox([pid_explain_button, pid_explain_output])
display(pid_explain_ui)


### 11.4 Concluding Remarks

In this notebook, the LLM serves only to translate a numerical description of the PID response into human-readable text. Python runs the simulation and calculates the measurements; the model receives those measurements together with a description of the system and generates an interpretation. It does not observe or control a physical plant.

The LLM is **outside the automatic control and decision process**. Each control input is calculated by the PID equation using the selected gains and the error signals. The model neither chooses the gains nor changes the target or control input. Its reply is displayed as text and is never fed back into the simulation. A missing or incorrect explanation therefore does not change the simulated response.

An explanation may still influence a person's choice of gains for a later experiment. That is an indirect role in human decision-making, rather than an automatic role in the feedback loop. The plots and calculated measurements remain the evidence against which the generated text must be checked. A fluent explanation can contain unsupported causal claims, and it does not establish stability or optimal tuning.

This example demonstrates a local AI interface for interpreting experimental results. Embedding AI into the controller, automatically tuning gains from its output, or allowing it to select control actions would require a different implementation and separate validation.


---
### 11.5 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Use Section 11.3 to explain two different gain combinations from the full PID sliders. Record the gains, overshoot, final error, and settling time for each, and quote a sentence from each LLM explanation. Do the explanations correctly describe the differences visible in the plots and measurements?**

(answer here)


**Check one LLM explanation critically. Identify a claim supported by a reported measurement and a causal explanation or stability claim that needs more evidence. If the model correctly avoids unsupported claims, give an example of that qualification. What additional experiment would help check its explanation, and why does asking for another explanation not train the model?**

(answer here)


**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)